# BPI2019 — 03 · End-to-end decoupled pipeline

Wires the re-engineered features into the full method: **cluster on behaviour → KPI-rank the cohorts →
attribute cohorts to static context**. Uses the choices validated in `01`/`02`:
- clustering on the 11 engineered behavioural features (z-scored, k=6, silhouette ≈ 0.48; no TF-IDF in the distance);
- classification with a tuned native-categorical XGBoost on **structural context only** (no vendor);
- SHAP driver analysis is deferred to `04`.

Artifacts are written to `../artifacts/`.

## 1. Run the full pipeline (raw XES → artifacts)

All logic lives in `../src/` (`data_preparation.py`, `data_modelling.py`, `orchestrator.py`); this
notebook just wires it together so the outputs can always be regenerated with one call. The heavy XES
parse is cached to parquet, so the first run is slow and every rerun is quick (pass
`force_reparse=True` to rebuild from the raw `.xes`).


In [1]:
import os
import sys

REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))  # repository root (parent of notebooks/)
SRC = os.path.join(REPO, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from orchestrator import run_pipeline

# force_reparse=False reuses the cached parquet parse of the raw XES; set True to rebuild from .xes
results = run_pipeline(repo_root=REPO, k=6, force_reparse=False, weighting="balanced")

  [prep] loaded cached parquet (set force=True to re-parse the XES)
  [prep] engineering behavioural + context features ...
  [model] clustered k=6 | silhouette 0.478
  [model] XGBoost [balanced] acc 0.655 | macro-F1 0.492 | weighted-F1 0.629
  [done] 71.4s | artifacts -> <project root>\artifacts
         cohort_labels.csv, cohort_kpis.csv, cohort_path_transitions.csv,
         cohort_top_variants.csv, shap_cluster_drivers.csv, classification_report.csv,
         pipeline_summary.json


## 2. Inspect the regenerated outputs

Quick sanity check on the artifacts written to `../artifacts/`. The narrative + visuals live in
`04_insights.ipynb`, which reads these same files.


In [3]:
import pandas as pd

s = results["summary"]
print(
    f"k={s['k']} | silhouette {s['silhouette']} | n_cases {s['n_cases']:,} | runtime {s['runtime_seconds']}s"
)
print(
    f"classification [{s['classification']['weighting']}]: "
    f"acc {s['classification']['accuracy']} | macro-F1 {s['classification']['macro_f1']} | "
    f"weighted-F1 {s['classification']['weighted_f1']}"
)

print("\nCohort KPIs (efficiency rank 1 = reference/benchmark):")
display(
    results["cohort_kpis"][
        [
            "n_cases",
            "pct",
            "cycle_time_days_median",
            "automation_rate",
            "rework_rate",
            "payment_block_rate",
            "avg_events",
            "eff_rank",
        ]
    ]
)

print("Top distinctive transition per cohort:")
display(
    results["transitions"]
    .sort_values(["cohort", "lift"], ascending=[True, False])
    .groupby("cohort")
    .head(1)[["cohort", "transition", "lift"]]
)

k=6 | silhouette 0.478 | n_cases 251,734 | runtime 71.4s
classification [balanced]: acc 0.655 | macro-F1 0.492 | weighted-F1 0.629

Cohort KPIs (efficiency rank 1 = reference/benchmark):


,n_cases,pct,cycle_time_days_median,automation_rate,rework_rate,payment_block_rate,avg_events,eff_rank
cohort,,,,,,,,
4,846,0.336,0.058,0.915,0.151,0.044,200.553,1
0,147762,58.698,66.059,0.258,0.105,0.000,5.358,2
5,51206,20.341,83.716,0.253,0.140,1.000,6.524,3
1,8751,3.476,3.087,0.078,1.000,0.004,2.678,4
3,28677,11.392,15.850,0.074,0.164,0.000,2.416,4
2,14492,5.757,85.719,0.422,0.655,0.315,14.335,4


Top distinctive transition per cohort:


,cohort,transition,lift
0,0,Record Invoice Receipt -> Clear Invoice,1.64
5,1,Create Purchase Order Item -> Delete Purchase ...,28.71
9,2,Record Service Entry Sheet -> Record Goods Rec...,14.31
14,3,Create Purchase Order Item -> Record Goods Rec...,3.20
19,4,Record Service Entry Sheet -> Record Service E...,64.11
24,5,Record Goods Receipt -> Remove Payment Block,4.75
